Testing FAKER, ran a simpel generated code to understand how it works.

In [37]:
import importlib
import testdata.generate
import pipeline.bronze

importlib.reload(testdata.generate)

from testdata.generate import generate_customers

customers = generate_customers(5)
for c in customers:
    print(c)

{'customer_id': 10000, 'name': 'Alva Fredriksson', 'email': 'folkeberg@example.net', 'city': 'Karlskoga'}
{'customer_id': 10001, 'name': 'Alfred Bengtsson', 'email': 'alexandraandersson@example.org', 'city': 'Mölndal'}
{'customer_id': 10002, 'name': 'Lennart Persson', 'email': 'nystromgustav@example.com', 'city': 'Karlstad'}
{'customer_id': 10003, 'name': 'Cornelia Olausson', 'email': 'backmandaniel@example.com', 'city': 'Malmö'}
{'customer_id': 10004, 'name': 'Kjell Westin', 'email': 'santonsson@example.net', 'city': 'Nyköping'}


In [38]:
# förväntat beteende
assert len(customers) == 5
assert set(customers[0].keys()) == {"customer_id", "name", "email", "city"}
assert len({c["customer_id"] for c in customers}) == 5, "ID:n ska vara unika"

# reproducerbarhet: samma seed ska ge exakt samma data
assert generate_customers(5, seed=42) == customers
assert generate_customers(5, seed=1) != customers


Still need to understand how certain things would work.
and limit test the FAKER dependency, aswell as the asserts being abit...
its not realy a wide net on the asserts, so i need to update this later, but for now it checks out as a simpel test.


In [39]:

importlib.reload(testdata.generate)
importlib.reload(pipeline.bronze)
from testdata.generate import generate_customers, build_delivery
from pipeline.bronze import extract_from_zip
from openpyxl import load_workbook
import io

customers = generate_customers(5)
zip_bytes = build_delivery(customers, "customers.xlsx")
print("bytes:", len(zip_bytes))

bytes: 4686


In [40]:
xlsx_bytes = extract_from_zip(zip_bytes, "customers.xlsx")
assert xlsx_bytes[:4] == b"PK\x03\x04"

ws = load_workbook(io.BytesIO(xlsx_bytes)).active
header = [cell.value for cell in ws[1]]

assert header == ["customer_id", "name", "email", "city"]
assert ws.max_row == 6, "1 rubrik + 5 rader"


Some more faker testing. creat zip, convert to xlsx, check contents, generate 5 customers
this means it will have 1 headline and 5 rows.
i convert it to bytes since its faster and more accurate, i just need to get AI to help me with the byte structure so i know what to grab. check the first assert, thats grabbing the "magic number" for what file type it is without caring for filename,
PK is the creator of the zip format, and x03+x04 is the local file header inside a zip.
Since i plan on using alot of different formats in the future, working with bytes is the fastest way and dividing it up into cells makes the most logical sense, even if i cant visualize the actual byte structure in my head to know what corresponds to what.

That issue is mostly solved by AI.

CSV, JSON, xlsx and zip all become bytes
so that my code wont have to care what the file actually is later down the line.

### Test: simulated delivery (rows → xlsx → zip → bytes)

Generates 5 customers with Faker, packs them as xlsx inside a zip (like
Retail II is delivered), then extracts and checks the result.

Expected: 1 header row + 5 data rows = 6 rows in total.

Everything is handled as `bytes` so the same functions work for any
format. The first assert checks the file signature (`PK\x03\x04`), which
proves the extracted file is a valid zip-based file (xlsx is one).

In [41]:
import importlib
import pipeline.convert
importlib.reload(pipeline.convert)

from pipeline.convert import xlsx_to_dataframe

customers = generate_customers(5)
xlsx_bytes_fake = extract_from_zip(build_delivery(customers, "customers.xlsx"), "customers.xlsx")


sheets_fake = xlsx_to_dataframe(xlsx_bytes_fake)
print({name: df.shape[0] for name, df in sheets_fake.items()})



{'Sheet': 5}


In [42]:
df_fake = sheets_fake["Sheet"] #Need the correct name for now from the Name
print(df_fake.dtypes)
df_fake

customer_id    int64
name             str
email            str
city             str
dtype: object


,customer_id,name,email,city
0,10000,Alva Fredriksson,folkeberg@example.net,Karlskoga
1,10001,Alfred Bengtsson,alexandraandersson@example.org,Mölndal
2,10002,Lennart Persson,nystromgustav@example.com,Karlstad
3,10003,Cornelia Olausson,backmandaniel@example.com,Malmö
4,10004,Kjell Westin,santonsson@example.net,Nyköping


In [43]:
assert list(sheets_fake) == ["Sheet"] #matching the name
assert df_fake.shape == (5,4) #matching row and col
assert list(df_fake.columns) == ["customer_id", "name", "email", "city"] #self explanatory
assert df_fake.isna().sum().sum() == 0 #check that neither row or col isna
